# ATS Vibration Report — Priority Mismatch Detector (fan reports)

Flags reports where the stated **Priority N** doesn't match what the report's own evidence implies — combining the **Recommendations/Comments text** and the **Spectrum** chart's actual tallest peak (read directly off its pixels, not a printed "Fund Amp" field), both checked against the **stated priority**.

Scope: one company's fan reports, with a Google Drive folder of manually-verified priorities to check against (see Section 3).

**How this works (v2 scope):**
- Extracts Recommendations/Comments/Equipment ID/Date Tested/Priority as native PDF text (no OCR needed for these fields).
- OCRs the embedded chart screenshot (Spectrum/Waterfall/Trend combined image) just to: detect and exclude "colored spectrum" style reports (per your instruction), read the Spectrum plot's unit (in/s, g, gE) off its title text, and read the sensor location/direction label (e.g. "Mtr Shaft H") off the same title text.
- Reads the **Spectrum** plot's tallest genuine peak directly from the chart image's pixels: finds the highest point of the actual trace line, ignoring on-chart markers that get in the way (highlighted fault-frequency bars, zoom-selection boxes) while still allowing small annotations sitting right on a real peak (a circle, a number) - then snaps that height down to the nearest y-axis label actually printed on the chart. See `ats_priority_checker/graph_signals.py` for exactly how markers are told apart from real data, and its module docstring for validation status. Priority thresholds from the peak amplitude (the in/s 3/4 boundary and the gE 1 boundary were refit against 799 real reports' stated priorities - see graph_signals.py's velocity_priority_hint/acceleration_enveloping_priority_hint docstrings for the derivation and what's still too data-thin to move):
  - **in/s (velocity):** >1 → 1, 0.5–1 → 2, 0.14–0.5 → 3, <0.14 → 4
  - **gE (acceleration enveloping):** >0.45 → 1, 0.3–0.45 → 2, 0.09–0.3 → 3, <0.09 → 4
  - **g (acceleration):** >2.5 → 1, >2 → 2, >1 → 3, else → 4
- Waterfall and Trend charts are **not** read (no pixel analysis on either), and there is currently **no cross-report escalation signal** - comparing a report against the same equipment's own prior dated test was tried and removed: this report set has multiple distinct measurement points per equipment (often in different units), and not every equipment_id has a same-point history to compare against, so there wasn't a reliably consistent per-machine timeline in this data. See `ats_priority_checker/graph_signals.py`'s module docstring for the full history before re-adding anything like this. Each report is judged only on its own text and its own chart reading.
- Uses a **frozen pretrained sentence-embedding model** (not fine-tuned, not trained from scratch) to turn text into vectors, then trains a small classifier on top to predict "what priority does this text imply." This is the right choice at this dataset size — a from-scratch deep model needs orders of magnitude more data.
- A report is flagged when: the text-implied priority disagrees with the stated one; OR the model has low confidence that the text supports the stated priority; OR the Spectrum peak reading implies a **more urgent** priority than stated.

Not implemented yet: comparing the *frequency* of each report's peak across a machine's dated history (not just its amplitude) to catch a resonance shifting frequency - see graph_signals.py's module docstring.

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pymupdf pillow scikit-learn sentence-transformers joblib pytesseract


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Get the code + point at your data

Clone this repo (so `ats_priority_checker` is importable), and set `PDF_DIR` to wherever your report PDFs live in Drive. The repo is public, so no credentials are needed.

Re-run this cell any time to pick up the latest code - it also clears any previously-imported copy of `ats_priority_checker` from this kernel, so a `git pull` here actually takes effect immediately. You no longer need to Restart session after a code update; just re-run this cell, then re-run whichever cells you need after it.


In [ ]:
import sys
from pathlib import Path

REPO_DIR = "/content/ats-fans-project"
REPO_URL = "https://github.com/r-chicken/ATS-Fans-Project.git"

if Path(REPO_DIR).exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Force Python to re-read our package from disk instead of reusing whatever
# was imported earlier in this session - without this, a `git pull` above
# updates the files but not the code you're actually running.
for _mod_name in list(sys.modules):
    if _mod_name == "ats_priority_checker" or _mod_name.startswith("ats_priority_checker."):
        del sys.modules[_mod_name]


In [ ]:
from pathlib import Path

PDF_DIR = Path("/content/drive/MyDrive/ATS_AI_Project/ATS_FanReports")          # <-- your fan report PDFs
OUT_DIR = Path("/content/drive/MyDrive/ATS_AI_Project/ats_claude_reports.csv")  # extracted CSVs + model saved here, persisted in Drive (a folder despite the name - build_dataset writes several CSVs into it)
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Extract text + Spectrum peak + filter report style from every PDF

This reads every PDF in `PDF_DIR`, parses out the fields, reads each report's Spectrum peak off its chart pixels, and writes three CSVs to `OUT_DIR`:
- `dataset.csv` — usable rows, parsed cleanly, waterfall-style. Includes `spectrum_unit`, `measurement_point` (sensor location/direction, e.g. "Mtr Shaft H"), `spectrum_peak_amplitude` (the pixel-read, axis-label-floored peak), and `spectrum_priority_hint`.
- `excluded_style.csv` — colored-spectrum-style rows, kept aside for reference
- `parse_errors.csv` — anything that didn't parse cleanly (review these — they usually mean the regex needs a small tweak for a report variant not seen yet)

`max_pages=1` below means only the first page of each PDF is read - some reports have a second page with nothing of note on it, so it's skipped entirely.

`filter_style=True`: style is detected by OCR'ing the chart screenshot and checking for the literal words "Colored Spectrum", not by an image-colorfulness heuristic - the earlier color-based version was only calibrated on 2 examples and started misclassifying real waterfall-style reports once tested against hundreds of real files. Text detection is slower per PDF (OCR isn't free) but tracks the actual thing that makes a report that style, so it won't drift as your report mix changes, and it stays useful even now that the colored-spectrum PDFs have been manually removed - it's your safety net if one slips back in later.


In [ ]:
from ats_priority_checker.dataset import build_dataset

summary = build_dataset(PDF_DIR, OUT_DIR, max_pages=1, filter_style=True)
summary


### Re-running this later after a code update

This step is slow because it OCRs every PDF's chart image AND reads the Spectrum peak's pixels. If a future code update only changes *logic* built on top of already-cached values (style detection, or the measurement_point label), you don't need to re-run the cell above and wait through OCR/pixel analysis again. Use this instead - it re-derives everything it can from what's already cached in `OUT_DIR` and finishes in seconds:

```python
from ats_priority_checker.dataset import recompute_dataset
summary = recompute_dataset(OUT_DIR, filter_style=True)
summary
```

What `recompute_dataset` CAN refresh from the cache alone (seconds, no re-reading PDFs):
- `style`, from the cached chart OCR text
- `measurement_point`, from the cached chart OCR text - use this after changing `graph_signals.detect_measurement_point()`

**Exception: the Spectrum peak reading itself.** `spectrum_peak_amplitude` / `spectrum_priority_hint` are read from the chart image's PIXELS (see `graph_signals.read_spectrum_peak`), not just its cached OCR text - a change to the peak-finding heuristics or the in/s, gE, g threshold functions always needs the slow cell above, same as changing PDF parsing itself would.


In [ ]:
import pandas as pd

errors = pd.read_csv(OUT_DIR / "parse_errors.csv")
print(f"{len(errors)} rows need review")
errors[["source_file", "page_number", "parse_notes"]].head(20)


## 3. Build a hand-labeling sheet

You need at least a subset of reports where a human has judged whether the priority actually matches the writeup — that's the only way to know if the detector is working, and the only way to measure precision/recall.

**Already have manually-verified priorities for this set?** (e.g. your ~115-report verification sheet from Drive) - you can skip straight to the "After you've labeled the sheet" cell below and upload that CSV directly, as long as it has a `report_id` column matching `dataset.csv`'s (`{source_file}_p{page_number}`) plus `human_label` (`match`/`mismatch`/`unsure`) and, for mismatches, a `human_notes` entry following the "Priority should be N because ..." convention. Otherwise, use this cell to export a blank sheet to fill in - start with ~100-150, you can label more later.

This exports a CSV with blank `human_label` (fill in `match`, `mismatch`, or `unsure`) and `human_notes` columns. Open it in Google Sheets, fill it in, then save it back (or update `LABELED_CSV` below to point at wherever you saved it).


In [ ]:
from ats_priority_checker.labeling import export_for_labeling

LABELING_CSV = OUT_DIR / "to_label_new.csv"
export_for_labeling(OUT_DIR / "dataset.csv", LABELING_CSV, sample_n=150)
print(f"Open and label: {LABELING_CSV}")


### After you've labeled the sheet

Upload your labeled CSV directly into this Colab session rather than pointing at a Drive path. Google Drive allows two files with the identical name in the same folder, so a re-uploaded/renamed file can silently collide with the original blank export and Colab's Drive mount can end up reading the wrong one (this is what caused "0 rows have a human label" before, even with real labels in the file) - uploading straight into the session sidesteps that entirely.


In [ ]:
from google.colab import files

uploaded = files.upload()  # pick your labeled CSV from your computer in the dialog
LABELED_CSV = list(uploaded.keys())[0]

check = pd.read_csv(LABELED_CSV)
print(f"{len(check)} rows. human_label counts: {dict(check['human_label'].value_counts())}")


In [ ]:
from ats_priority_checker.labeling import merge_labels

labeled_df = merge_labels(OUT_DIR / "dataset.csv", LABELED_CSV, OUT_DIR / "dataset_with_labels.csv")


## 4. Train the text -> priority model

Trains on `true_priority` as the target, not the raw stated priority. For match rows (and any unlabeled report) those are the same thing - stated priority is the best available answer. But for your 11 mismatch rows, `true_priority` is the corrected number you wrote in human_notes ("Priority should be N because ...") - training on the stated priority for those rows would teach the model to defend the exact errors you're trying to catch, so the correction is used instead wherever you provided one.


In [ ]:
from ats_priority_checker.model import report_text, embed_texts, cross_validated_predictions

df = pd.read_csv(OUT_DIR / "dataset_with_labels.csv")
df = df.dropna(subset=["true_priority"]).reset_index(drop=True)

texts = df.apply(report_text, axis=1).tolist()
X = embed_texts(texts)
y = df["true_priority"].to_numpy()

cv_result = cross_validated_predictions(X, y, n_splits=5)
print(f"used {cv_result['n_splits']}-fold CV, classes seen: {cv_result['classes']}")


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y, cv_result["pred"]))


## 5. Flag mismatches, then validate against your hand labels

`flagged` has one row per report with `predicted_priority`, `confidence_in_stated_priority`, and `flag_mismatch`. Where you have a human label, compare directly — that precision/recall is the real signal for whether this prototype is usable, not the classification report above.


In [ ]:
from ats_priority_checker.model import flag_mismatches

flagged = flag_mismatches(df, cv_result["pred"], cv_result["proba"], cv_result["classes"])
flagged.to_csv(OUT_DIR / "flagged.csv", index=False)
flagged[["report_id", "priority_num", "predicted_priority", "confidence_in_stated_priority", "flag_mismatch", "flag_reason"]].head(20)


In [ ]:
labeled_subset = flagged.dropna(subset=["human_label"])
labeled_subset = labeled_subset[labeled_subset["human_label"] != ""]

if len(labeled_subset) == 0:
    print("No human labels yet - go label some reports in Section 3 to see real precision/recall here.")
else:
    from sklearn.metrics import precision_score, recall_score, confusion_matrix

    y_true = (labeled_subset["human_label"] == "mismatch").astype(int)
    y_pred = labeled_subset["flag_mismatch"].astype(int)
    print(f"n labeled = {len(labeled_subset)}")
    print(f"precision = {precision_score(y_true, y_pred, zero_division=0):.2f}")
    print(f"recall    = {recall_score(y_true, y_pred, zero_division=0):.2f}")
    print(confusion_matrix(y_true, y_pred))


### How often does the model land on the *right* corrected priority?

Precision/recall above only checks whether a mismatch got flagged at all. This checks something stricter: of the mismatch rows where you wrote a "Priority should be N" correction, how often did the model's predicted priority land exactly on your corrected number, not just disagree with the stated one.


In [ ]:
corrected_subset = flagged.dropna(subset=["corrected_priority"])

if len(corrected_subset) == 0:
    print("No corrected-priority notes yet - see Section 3.")
else:
    exact_match = (corrected_subset["predicted_priority"] == corrected_subset["corrected_priority"])
    print(f"n with a correction = {len(corrected_subset)}")
    print(f"predicted priority exactly matched your correction: {exact_match.sum()} / {len(corrected_subset)}")
    corrected_subset[["report_id", "priority_num", "corrected_priority", "predicted_priority"]]


## 6. Save the model (for scoring new reports later)

Fits on 100% of current data and saves to Drive. When you add the +200 reports (or more later), just rerun Sections 2 and 4-6 on the combined folder — same pipeline, bigger dataset.


In [ ]:
from ats_priority_checker.model import fit_final_model, save_bundle

final_clf = fit_final_model(X, y)
save_bundle(final_clf, OUT_DIR / "model" / "priority_classifier.joblib")
print("saved model to", OUT_DIR / "model" / "priority_classifier.joblib")


## 7. Scoring brand-new reports later

Once you have a saved model, score new PDFs without retraining:


In [ ]:
from ats_priority_checker.dataset import build_dataset
from ats_priority_checker.model import load_bundle, embed_texts, report_text

NEW_PDF_DIR = Path("/content/drive/MyDrive/ats_reports/new_pdfs")   # e.g. next month's batch
NEW_OUT_DIR = Path("/content/drive/MyDrive/ats_reports/new_out")

build_dataset(NEW_PDF_DIR, NEW_OUT_DIR)
new_df = pd.read_csv(NEW_OUT_DIR / "dataset.csv")

clf, embedding_model_name = load_bundle(OUT_DIR / "model" / "priority_classifier.joblib")
new_X = embed_texts(new_df.apply(report_text, axis=1).tolist(), model_name=embedding_model_name)

new_df["predicted_priority"] = clf.predict(new_X)
new_df["flag_mismatch"] = new_df["predicted_priority"] != new_df["priority_num"]
new_df.to_csv(NEW_OUT_DIR / "flagged.csv", index=False)
new_df[new_df["flag_mismatch"]][["report_id", "priority_num", "predicted_priority"]]
